# Credit Default Prediction — Modeling

This notebook trains and evaluates machine learning models for predicting loan default probability using the client-level features created during feature engineering.

The modeling workflow includes:
- loading the prepared feature dataset;
- creating a time-aware train-validation split;
- training Logistic Regression as a baseline;
- training Random Forest as a non-linear tree-based alternative;
- training the final CatBoost model;
- comparing model performance using ROC AUC;
- evaluating temporal robustness on an additional out-of-time period.

The primary evaluation metric is ROC AUC, following the original competition setup.

## 1. Importing Libraries and Loading Data

The prepared dataset contains one row per application, engineered credit-history features, and the binary target variable `flag`.

The application identifier `id` is stored as the DataFrame index. According to the dataset description, larger application IDs correspond to later application dates, which allows the chronological order of applications to be preserved during validation.

In [1]:
from pathlib import Path
import gc

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", None)

In [2]:
PROJECT_ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())

FEATURES_PATH = (PROJECT_ROOT / "data" / "processed" / "train_features.parquet")

RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = PROJECT_ROOT / "models"
SNAPSHOT_DIR = RESULTS_DIR / "catboost_snapshots"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
df = pd.read_parquet(FEATURES_PATH)

print(f"Shape: {df.shape}")

df.head()

Shape: (3000000, 548)


,credit_count,ever_loans5,share_loans5,ever_loans530,share_loans530,ever_loans3060,share_loans3060,ever_loans6090,share_loans6090,ever_loans90,share_loans90,is_zero_loans5_first,is_zero_loans530_first,is_zero_loans3060_first,is_zero_loans6090_first,is_zero_loans90_first,is_zero_util_first,is_zero_over2limit_first,is_zero_maxover2limit_first,enc_loans_account_cur_first,enc_loans_credit_type_first,enc_loans_account_holder_type_first,enc_loans_credit_status_first,pre_since_opened_first,pre_since_confirmed_first,pre_pterm_first,pre_fterm_first,pre_till_pclose_first,pre_till_fclose_first,pre_loans_credit_limit_first,pre_loans_next_pay_summ_first,pre_loans_outstanding_first,pre_loans_total_overdue_first,pre_loans_max_overdue_sum_first,pre_loans_credit_cost_rate_first,pre_loans5_first,pre_loans530_first,pre_loans3060_first,pre_loans6090_first,pre_loans90_first,pre_util_first,pre_over2limit_first,pre_maxover2limit_first,is_zero_loans5_last,is_zero_loans530_last,is_zero_loans3060_last,is_zero_loans6090_last,is_zero_loans90_last,is_zero_util_last,is_zero_over2limit_last,is_zero_maxover2limit_last,enc_loans_account_cur_last,enc_loans_credit_type_last,enc_loans_account_holder_type_last,enc_loans_credit_status_last,pre_since_opened_last,pre_since_confirmed_last,pre_pterm_last,pre_fterm_last,pre_till_pclose_last,pre_till_fclose_last,pre_loans_credit_limit_last,pre_loans_next_pay_summ_last,pre_loans_outstanding_last,pre_loans_total_overdue_last,pre_loans_max_overdue_sum_last,pre_loans_credit_cost_rate_last,pre_loans5_last,pre_loans530_last,pre_loans3060_last,pre_loans6090_last,pre_loans90_last,pre_util_last,pre_over2limit_last,pre_maxover2limit_last,enc_loans_account_cur_nunique,enc_loans_credit_type_nunique,enc_loans_account_holder_type_nunique,enc_loans_credit_status_nunique,enc_loans_account_cur_share_0,enc_loans_account_cur_share_1,enc_loans_account_cur_share_2,enc_loans_account_cur_share_3,enc_loans_credit_type_share_0,enc_loans_credit_type_share_1,enc_loans_credit_type_share_2,enc_loans_credit_type_share_3,enc_loans_credit_type_share_4,enc_loans_credit_type_share_5,enc_loans_account_holder_type_share_0,enc_loans_account_holder_type_share_1,enc_loans_account_holder_type_share_2,enc_loans_account_holder_type_share_3,enc_loans_account_holder_type_share_4,enc_loans_account_holder_type_share_5,enc_loans_account_holder_type_share_6,enc_loans_credit_status_share_0,enc_loans_credit_status_share_1,enc_loans_credit_status_share_2,enc_loans_credit_status_share_3,enc_loans_credit_status_share_4,enc_loans_credit_status_share_5,enc_loans_credit_status_share_6,pre_since_opened_nunique,pre_since_confirmed_nunique,pre_pterm_nunique,pre_fterm_nunique,pre_till_pclose_nunique,pre_till_fclose_nunique,pre_loans_credit_limit_nunique,pre_loans_next_pay_summ_nunique,pre_loans_outstanding_nunique,pre_loans_total_overdue_nunique,pre_loans_max_overdue_sum_nunique,pre_loans_credit_cost_rate_nunique,pre_loans5_nunique,pre_loans530_nunique,pre_loans3060_nunique,pre_loans6090_nunique,pre_loans90_nunique,pre_util_nunique,pre_over2limit_nunique,pre_maxover2limit_nunique,pre_since_opened_share_0,pre_since_opened_share_1,pre_since_opened_share_2,pre_since_opened_share_3,pre_since_opened_share_4,pre_since_opened_share_5,pre_since_opened_share_6,pre_since_opened_share_7,pre_since_opened_share_8,pre_since_opened_share_9,pre_since_opened_share_10,pre_since_opened_share_11,pre_since_opened_share_12,pre_since_opened_share_13,pre_since_opened_share_14,pre_since_opened_share_15,pre_since_opened_share_16,pre_since_opened_share_17,pre_since_opened_share_18,pre_since_opened_share_19,pre_since_confirmed_share_0,pre_since_confirmed_share_1,pre_since_confirmed_share_2,pre_since_confirmed_share_3,pre_since_confirmed_share_4,pre_since_confirmed_share_5,pre_since_confirmed_share_6,pre_since_confirmed_share_7,pre_since_confirmed_share_8,pre_since_confirmed_share_9,pre_since_confirmed_share_10,pre_since_confirmed_share_11,pre_since_confirmed_share_12,pre_since_confirmed_share_13,pre_since_

## 2. Time-Aware Train-Validation Split

Application IDs preserve chronological order: larger `id` values correspond to later application dates.

Before defining the train-validation boundary, we examine whether the target distribution changes across the chronological sequence of applications. This helps identify potential temporal shifts in default rates and informs the validation strategy.

In [4]:
target_by_period = (df.assign(period=pd.qcut(df.index, q=10, labels=False))
                      .groupby("period")
                      .agg(applications=("flag", "size"),
                           default=("flag", "sum"),
                           default_rate=("flag", "mean")))
target_by_period["default_rate"] *= 100

target_by_period

,applications,default,default_rate
period,,,
0,300000,9368,3.122667
1,300000,10102,3.367333
2,300000,10551,3.517000
3,300000,11988,3.996000
4,300000,11313,3.771000
5,300000,11672,3.890667
6,300000,11208,3.736000
7,300000,10337,3.445667
8,300000,10324,3.441333


### 2.1 Defining the Validation Period

The default rate varies across the chronological sequence of applications, indicating a temporal shift in the target distribution.

To simulate prediction on future applications, the earliest 80% of applications are used for training and the latest 20% are reserved for validation.

This produces an out-of-time validation set of approximately 600,000 applications while preserving the chronological ordering of the data.

In [5]:
df = df.sort_index()
split_idx = int(len(df) * 0.8)

train_df = df.iloc[:split_idx].copy()
valid_df = df.iloc[split_idx:].copy()

print(f"Train shape: {train_df.shape}")
print(f"Validation shape: {valid_df.shape}")

Train shape: (2400000, 548)
Validation shape: (600000, 548)


In [6]:
split_summary = pd.DataFrame({"applications": [len(train_df), len(valid_df)],
                              "min_id": [train_df.index.min(), valid_df.index.min()],
                              "max_id": [train_df.index.max(), valid_df.index.max()],
                              "defaults": [train_df["flag"].sum(), valid_df["flag"].sum()],
                              "default_rate": [train_df["flag"].mean() * 100, valid_df["flag"].mean() * 100]}, index=["Train", "Validation"])
split_summary

,applications,min_id,max_id,defaults,default_rate
Train,2400000,0,2399999,86539,3.605792
Validation,600000,2400000,2999999,19903,3.317167


### 2.2 Separating Features and Target

The target variable `flag` is separated from the engineered features for both training and validation datasets.

The application identifier `id` remains in the DataFrame index and is not used as a predictive feature.

In [7]:
X_train = train_df.drop(columns="flag")
y_train = train_df["flag"]

X_valid = valid_df.drop(columns="flag")
y_valid = valid_df["flag"]

print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"X_valid: {X_valid.shape}")
print(f"y_valid: {y_valid.shape}")

X_train: (2400000, 547)
y_train: (2400000,)
X_valid: (600000, 547)
y_valid: (600000,)


## 3. Feature Types for Modeling

The engineered dataset contains both numerical features and encoded categorical features.

Some original variables use arbitrary numerical codes to represent categories or binarized intervals. These codes should not be interpreted as continuous numerical values, because their numerical order and distance do not carry quantitative meaning.

Therefore, the features are separated into numerical and categorical groups before modeling.

In [8]:
categorical_features = [col for col in X_train.columns if col.endswith("_first") or col.endswith("_last")]

numerical_features = [col for col in X_train.columns if col not in categorical_features]

print(f"Categorical features: {len(categorical_features)}")
print(f"Numerical features: {len(numerical_features)}")
print(f"Total: {len(categorical_features) + len(numerical_features)}")

Categorical features: 64
Numerical features: 483
Total: 547


## 4. Baseline Model — Logistic Regression

Logistic Regression is used as a baseline to establish a reference performance level before training more flexible non-linear models.

Encoded categorical features are one-hot encoded because their numerical codes do not represent meaningful numerical order. Numerical features are standardized before fitting the model.

All preprocessing steps are fitted only on the training data and then applied to the validation data to avoid information leakage.

### 4.1 Preprocessing Pipeline

Categorical features are transformed using one-hot encoding, while numerical features are standardized.

The preprocessing steps are combined using a `ColumnTransformer`, ensuring that each feature group receives the appropriate transformation.

In [9]:
preprocessor = ColumnTransformer(transformers = [("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
                                                 ("numerical", StandardScaler(), numerical_features)])

### 4.2 Training the Baseline Model

The dataset is highly imbalanced, with default cases representing only a small proportion of applications.

Since ROC AUC evaluates the ranking of predicted probabilities rather than classification accuracy at a fixed threshold, the baseline Logistic Regression model is initially trained using the natural class distribution without class weighting.

This provides a simple reference model against which more flexible models can be compared.

In [10]:
logistic_model = Pipeline(steps=[("preprocessor", preprocessor),
                                 ("classifier", LogisticRegression(max_iter=1000,
                                                                   solver="lbfgs"))])
logistic_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numerical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float

In [11]:
logistic_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](547,)","['credit_count','ever_loans5','share_loans5',...,'pre_loans530_share_8', 'pre_loans530_share_9','pre_loans530_share_17']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,547
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numerical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (

### 4.3 Evaluating the Baseline Model

The trained model is evaluated on the out-of-time validation set.

Since the competition metric is ROC AUC, we use the predicted probability of the positive class (`flag = 1`) rather than binary class predictions.

In [12]:
y_valid_pred = logistic_model.predict_proba(X_valid)[:, 1]
y_valid_pred[:10]

array([0.0198055 , 0.01031055, 0.02116296, 0.01707453, 0.05203659,
       0.01196465, 0.04366584, 0.02607358, 0.01702549, 0.01608382])

In [13]:
logistic_auc = roc_auc_score(y_valid, y_valid_pred)

print(f"Logistic Regression ROC AUC: {logistic_auc:.4f}")

Logistic Regression ROC AUC: 0.7267


The baseline Logistic Regression model achieves a ROC AUC of **0.7267** on the out-of-time validation set.

This result serves as a reference point for evaluating more flexible non-linear models.

## 5. Random Forest Model

Logistic Regression provides a linear baseline, but relationships between credit-history features and default risk may be non-linear and may involve interactions between multiple variables.

Random Forest is used as a second modeling approach. It combines multiple decision trees trained on different bootstrap samples and feature subsets, allowing the model to capture non-linear relationships and feature interactions without requiring feature scaling.

Categorical features are one-hot encoded because their numerical codes do not represent meaningful numerical order.

### 5.1 Preprocessing

Random Forest does not require numerical feature scaling because decision trees split features based on thresholds rather than distances or coefficient magnitudes.

Categorical features are one-hot encoded to prevent arbitrary category codes from being interpreted as ordered numerical values. Numerical features are passed through without transformation.

In [14]:
rf_preprocessor = ColumnTransformer(transformers=[("categorical", OneHotEncoder(handle_unknown="ignore",
                                                                                dtype=np.float32),
                                                                  categorical_features),
                                                  ("numerical", "passthrough", numerical_features)])

### 5.2 Training the Random Forest Model

Random Forest is trained as a non-linear tree-based alternative to the Logistic Regression baseline.

The model uses multiple decision trees trained on bootstrap samples of the training data. At each split, only a subset of features is considered, which introduces diversity between trees and reduces the tendency of individual decision trees to overfit.

In [15]:
random_forest_model = Pipeline(steps=[("preprocessor", rf_preprocessor),
                                      ("classifier", RandomForestClassifier(n_estimators=300,
                                                                            max_depth=12,
                                                                            min_samples_leaf=20,
                                                                            max_features="sqrt",
                                                                            n_jobs=-1,
                                                                            random_state=42))])
random_forest_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numerical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float

In [16]:
random_forest_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](547,)","['credit_count','ever_loans5','share_loans5',...,'pre_loans530_share_8', 'pre_loans530_share_9','pre_loans530_share_17']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,547
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...), ('numerical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (

### 5.3 Evaluating the Random Forest Model

The trained Random Forest model is evaluated on the same out-of-time validation set used for Logistic Regression.

Predicted probabilities for the positive class (`flag = 1`) are used to calculate ROC AUC, ensuring a consistent comparison between the models.

In [17]:
y_valid_pred_rf = random_forest_model.predict_proba(X_valid)[:,1]

random_forest_auc = roc_auc_score(y_valid, y_valid_pred_rf)

print(f"Random Forest ROC AUC: {random_forest_auc:.4f}")

Random Forest ROC AUC: 0.7070


The Random Forest model achieves a ROC AUC of **0.7070** on the out-of-time validation set.

Despite its ability to capture non-linear relationships and feature interactions, the model performs below the Logistic Regression baseline. This indicates that simply introducing a more flexible tree-based model does not necessarily improve predictive performance for the current feature representation.

The result motivates the use of gradient boosting, where trees are built sequentially to correct the errors of previous trees.

## 6. CatBoost Model

CatBoost is used as the final gradient boosting model.

Unlike Logistic Regression and Random Forest, CatBoost can work with categorical features directly. This allows encoded categorical variables to be treated as categories rather than as ordered numerical values.

Several hyperparameter configurations were evaluated during model development. To keep the final notebook concise and focused on the final modeling pipeline, the intermediate tuning experiments are omitted here.

The selected configuration is:

- `depth = 7`
- `learning_rate = 0.08`
- `l2_leaf_reg = 7`
- up to `2000` boosting iterations

The model is trained and evaluated using the same out-of-time split as the baseline models.

In [18]:
del logistic_model
del random_forest_model
del y_valid_pred
del y_valid_pred_rf

gc.collect()

114

In [19]:
X_train_cb = X_train
X_valid_cb = X_valid

for col in categorical_features:
    X_train_cb[col] = X_train_cb[col].astype(str)
    X_valid_cb[col] = X_valid_cb[col].astype(str)

### 6.1 Training the CatBoost Model

The selected CatBoost configuration is trained on the first 80% of applications and evaluated on the latest 20%.

ROC AUC is monitored on the validation set. Early stopping is used to stop training when additional boosting iterations no longer improve validation performance.

In [20]:
catboost_model = CatBoostClassifier(iterations=2000,
                                    learning_rate=0.08,
                                    depth=7,
                                    l2_leaf_reg=7,
                                    eval_metric="AUC",
                                    loss_function="Logloss",
                                    random_seed=42,
                                    verbose=100,
                                    save_snapshot=True,
                                    snapshot_file=str(SNAPSHOT_DIR / "catboost_training.cbsnapshot"),
                                    snapshot_interval=600)

In [21]:
catboost_model.fit(X_train_cb,
                   y_train,
                   cat_features=categorical_features,
                   eval_set=(X_valid_cb, y_valid),
                   early_stopping_rounds=150,
                   verbose=100)

0:	test: 0.5574932	best: 0.5574932 (0)	total: 5.02s	remaining: 2h 47m 16s
100:	test: 0.7331491	best: 0.7331491 (100)	total: 6m 58s	remaining: 2h 11m 13s
200:	test: 0.7405553	best: 0.7405553 (200)	total: 13m 49s	remaining: 2h 3m 40s
300:	test: 0.7433484	best: 0.7433484 (300)	total: 20m 17s	remaining: 1h 54m 31s
400:	test: 0.7449460	best: 0.7449460 (400)	total: 26m 56s	remaining: 1h 47m 27s
500:	test: 0.7461796	best: 0.7461796 (500)	total: 33m 42s	remaining: 1h 40m 51s
600:	test: 0.7471660	best: 0.7471660 (600)	total: 40m 54s	remaining: 1h 35m 13s
700:	test: 0.7477660	best: 0.7477660 (700)	total: 48m 12s	remaining: 1h 29m 20s
800:	test: 0.7482070	best: 0.7482070 (800)	total: 55m 17s	remaining: 1h 22m 46s
900:	test: 0.7485655	best: 0.7485657 (899)	total: 1h 2m 35s	remaining: 1h 16m 21s
1000:	test: 0.7489232	best: 0.7489232 (1000)	total: 1h 9m 24s	remaining: 1h 9m 15s
1100:	test: 0.7489855	best: 0.7489940 (1095)	total: 1h 16m 18s	remaining: 1h 2m 18s
1200:	test: 0.7490407	best: 0.7491521 (

CatBoostClassifier(depth=7, eval_metric='AUC', iterations=2000, l2_leaf_reg=7, learning_rate=0.08, loss_function='Logloss', random_seed=42, save_snapshot=True, snapshot_file='C:\\Study\\SkillBox\\results\\catboost_snapshots\\catboost_training.cbsnapshot', snapshot_interval=600, verbose=100)

### 6.2 Evaluating the CatBoost Model

The final CatBoost model is evaluated on the same out-of-time validation set as Logistic Regression and Random Forest.

In [22]:
y_valid_pred_cb = catboost_model.predict_proba( X_valid_cb)[:, 1]

catboost_auc = roc_auc_score(y_valid, y_valid_pred_cb)

print(f"CatBoost ROC AUC: {catboost_auc:.6f}")

CatBoost ROC AUC: 0.750233


In [ ]:
FINAL_MODEL_PATH = (MODELS_DIR / "credit_default_catboost.cbm")

catboost_model.save_model(FINAL_MODEL_PATH)

## 7. Temporal Robustness Check

The final feature set and CatBoost configuration were selected using the main out-of-time validation period.

To evaluate whether model performance remains stable across time, an additional earlier out-of-time validation period is used.

The model is trained on the first 70% of applications and evaluated on the following 10% of applications.

No feature engineering or hyperparameter tuning is performed using this secondary validation period.

In [24]:
del X_train_cb
del X_valid_cb
del X_train
del X_valid

del train_df
del valid_df
del df

del y_train
del y_valid
del y_valid_pred_cb
del catboost_model

gc.collect()

0

In [25]:
df_temporal = pd.read_parquet(FEATURES_PATH)
df_temporal = df_temporal.sort_index()

n = len(df_temporal)

train_end = int(n * 0.70)
validation_end = int(n * 0.80)

train_temporal_df = df_temporal.iloc[:train_end]
secondary_valid_df = df_temporal.iloc[train_end:validation_end]

print(f"Train: {train_temporal_df.shape}")
print(f"Secondary validation: {secondary_valid_df.shape}")
print(f"Train IDs: {train_temporal_df.index.min()} - {train_temporal_df.index.max()}")
print(f"Validation IDs: {secondary_valid_df.index.min()} - {secondary_valid_df.index.max()}")

Train: (2100000, 548)
Secondary validation: (300000, 548)
Train IDs: 0 - 2099999
Validation IDs: 2100000 - 2399999


In [26]:
X_train_temporal = train_temporal_df.drop(columns="flag")
y_train_temporal = train_temporal_df["flag"]

X_secondary_valid = secondary_valid_df.drop(columns="flag")
y_secondary_valid = secondary_valid_df["flag"]

print(f"X_train_temporal: {X_train_temporal.shape}")
print(f"X_secondary_valid: {X_secondary_valid.shape}")

print(f"Train default rate: {y_train_temporal.mean() * 100:.4f}%")
print(f"Secondary validation default rate: {y_secondary_valid.mean() * 100:.4f}%")

X_train_temporal: (2100000, 547)
X_secondary_valid: (300000, 547)
Train default rate: 3.6287%
Secondary validation default rate: 3.4457%


In [27]:
X_train_temporal_cb = X_train_temporal
X_secondary_valid_cb = X_secondary_valid

for col in categorical_features:
    X_train_temporal_cb[col] = X_train_temporal_cb[col].astype(str)
    X_secondary_valid_cb[col] = X_secondary_valid_cb[col].astype(str)

In [28]:
del df_temporal
del train_temporal_df
del secondary_valid_df
del X_train_temporal
del X_secondary_valid

gc.collect()

0

### 7.1 Training on the Secondary Temporal Split

The same CatBoost hyperparameters selected for the final model are used without modification.

Only the chronological training and validation periods are changed. This allows us to evaluate whether the model performance is stable across different time periods.

In [29]:
temporal_model = CatBoostClassifier(iterations=2000,
                                    learning_rate=0.08,
                                    depth=7,
                                    l2_leaf_reg=7,
                                    eval_metric="AUC",
                                    loss_function="Logloss",
                                    random_seed=42,
                                    verbose=100,
                                    save_snapshot=True,
                                    snapshot_file=str(SNAPSHOT_DIR / "catboost_temporal_check.cbsnapshot"),
                                    snapshot_interval=600)

In [30]:
temporal_model.fit(X_train_temporal_cb,
                   y_train_temporal,
                   cat_features=categorical_features,
                   eval_set=(X_secondary_valid_cb,
                             y_secondary_valid),
                   early_stopping_rounds=150,
                   verbose=100)

0:	test: 0.5759161	best: 0.5759161 (0)	total: 3.44s	remaining: 1h 54m 32s
100:	test: 0.7480142	best: 0.7480142 (100)	total: 6m	remaining: 1h 53m 7s
200:	test: 0.7547822	best: 0.7547822 (200)	total: 11m 50s	remaining: 1h 45m 56s
300:	test: 0.7571858	best: 0.7571858 (300)	total: 17m 28s	remaining: 1h 38m 38s
400:	test: 0.7590531	best: 0.7590531 (400)	total: 23m 10s	remaining: 1h 32m 24s
500:	test: 0.7604255	best: 0.7604255 (500)	total: 28m 52s	remaining: 1h 26m 22s
600:	test: 0.7612256	best: 0.7612256 (600)	total: 34m 32s	remaining: 1h 20m 25s
700:	test: 0.7621338	best: 0.7621338 (700)	total: 40m 15s	remaining: 1h 14m 35s
800:	test: 0.7626042	best: 0.7626127 (798)	total: 45m 55s	remaining: 1h 8m 44s
900:	test: 0.7631516	best: 0.7631516 (900)	total: 51m 39s	remaining: 1h 3m 1s
1000:	test: 0.7635264	best: 0.7635531 (994)	total: 57m 21s	remaining: 57m 14s
1100:	test: 0.7638590	best: 0.7638716 (1093)	total: 1h 3m 3s	remaining: 51m 29s
1200:	test: 0.7641315	best: 0.7641425 (1199)	total: 1h 8m

CatBoostClassifier(depth=7, eval_metric='AUC', iterations=2000, l2_leaf_reg=7, learning_rate=0.08, loss_function='Logloss', random_seed=42, save_snapshot=True, snapshot_file='C:\\Study\\SkillBox\\results\\catboost_snapshots\\catboost_temporal_check.cbsnapshot', snapshot_interval=600, verbose=100)

### 7.2 Temporal Robustness Result

In [31]:
y_secondary_pred = temporal_model.predict_proba(X_secondary_valid_cb)[:, 1]

temporal_auc = roc_auc_score(y_secondary_valid, y_secondary_pred)

print(f"Secondary temporal ROC AUC: {temporal_auc:.6f}")
print(f"Main validation ROC AUC: {catboost_auc:.6f}")
print(f"Difference: {temporal_auc - catboost_auc:+.6f}")

Secondary temporal ROC AUC: 0.765399
Main validation ROC AUC: 0.750233
Difference: +0.015166


In [32]:
temporal_results = pd.DataFrame([{"evaluation": "main_out_of_time_validation",
                                  "train_period": "first 80%",
                                  "validation_period": "last 20%",
                                  "roc_auc": catboost_auc},
                                 {"evaluation": "secondary_temporal_validation",
                                  "train_period": "first 70%",
                                  "validation_period": "70%-80%",
                                  "roc_auc": temporal_auc}])

temporal_results.to_csv(RESULTS_DIR / "temporal_validation_results.csv", index=False)
temporal_results

,evaluation,train_period,validation_period,roc_auc
0,main_out_of_time_validation,first 80%,last 20%,0.750233
1,secondary_temporal_validation,first 70%,70%-80%,0.765399


The final CatBoost configuration was additionally evaluated on an earlier out-of-time period using the same feature engineering approach and the same fixed hyperparameters.

- Main out-of-time validation ROC AUC: **0.7502**
- Secondary temporal validation ROC AUC: **0.7654**

The model therefore exceeded the target ROC AUC of **0.75** on two different temporal validation periods.

Performance was higher on the earlier validation period, suggesting that the most recent applications represent a more difficult prediction period and that some temporal distribution shift may be present.

No additional feature engineering or hyperparameter tuning was performed using the secondary temporal validation period.

## 8. Model Comparison and Conclusions

Three modeling approaches were evaluated using the same main out-of-time validation period.

Logistic Regression provides a linear baseline, while Random Forest captures non-linear relationships and feature interactions.

CatBoost achieves the highest ROC AUC and exceeds the target threshold of **0.75**.

An additional temporal robustness check is used to evaluate whether the selected CatBoost configuration maintains its predictive performance across another chronological validation period.

In [35]:
model_results = pd.DataFrame([{"model": "Logistic Regression",
                               "roc_auc": logistic_auc},
                              {"model": "Random Forest",
                               "roc_auc": random_forest_auc},
                              {"model": "CatBoost",
                               "roc_auc": catboost_auc}])

model_results = model_results.sort_values("roc_auc", ascending=False).reset_index(drop=True)
model_results.to_csv(RESULTS_DIR / "model_results.csv", index=False)
model_results

,model,roc_auc
0,CatBoost,0.750233
1,Logistic Regression,0.726678
2,Random Forest,0.707029


### Final Conclusions

The final CatBoost model achieved a ROC AUC of **0.7502** on the main out-of-time validation set, exceeding the project target of **0.75**.

The same CatBoost configuration achieved a ROC AUC of **0.7654** on an earlier secondary temporal validation period, providing additional evidence that the model retains predictive performance across different chronological periods.

Among the evaluated approaches, CatBoost performed best, outperforming both Logistic Regression and Random Forest.

The final feature set incorporates recent credit-history behavior alongside full-history aggregates. Recent delinquency, utilization, over-limit, and closure patterns provide additional information about a client's current credit behavior that may be diluted when the entire credit history is represented only through global aggregates.

The lower performance on the most recent validation period suggests the presence of temporal distribution shift and highlights the importance of time-aware validation for credit-risk modeling.